In [1]:
%pip install google-cloud-bigquery db-dtypes pandas-gbq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from google.cloud import bigquery
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Connect to BigQuery 
PROJECT_ID = "symbolic-folio-491316-s8"
client = bigquery.Client(project=PROJECT_ID)

# 2. Query your dbt-transformed features
query = f"""
    SELECT * 
    FROM `{PROJECT_ID}.raw_financial_data.fct_stock_ml_features` 
    ORDER BY price_date ASC
"""

# 3. Download directly into a Pandas DataFrame
print("Downloading data from BigQuery...")
df = client.query(query).to_dataframe()

print(f"Data shape: {df.shape}")

# 4. View the first 5 rows!
df.head()

c:\Users\abhay\anaconda3\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


c:\Users\abhay\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Data shape: (102, 8)


,price_date,symbol,close_price,volume,daily_return,ma_5,ma_20,target_direction
0,2026-03-04,AAPL,262.52,39803119,NaN,262.5200,262.5200,0
1,2026-03-05,AAPL,260.29,49658626,-0.008495,261.4050,261.4050,0
2,2026-03-06,AAPL,257.46,41120042,-0.010872,260.0900,260.0900,1
3,2026-03-09,AAPL,259.88,38218533,0.009400,260.0375,260.0375,1
4,2026-03-10,AAPL,260.83,30590765,0.003656,260.1960,260.1960,0


In [3]:
df.isnull().sum()

price_date          0
symbol              0
close_price         0
volume              0
daily_return        1
ma_5                0
ma_20               0
target_direction    0
dtype: int64

In [4]:
df.dropna()
df.drop_duplicates()

,price_date,symbol,close_price,volume,daily_return,ma_5,ma_20,target_direction
0,2026-03-04,AAPL,262.52,39803119,NaN,262.5200,262.5200,0
1,2026-03-05,AAPL,260.29,49658626,-0.008495,261.4050,261.4050,0
2,2026-03-06,AAPL,257.46,41120042,-0.010872,260.0900,260.0900,1
3,2026-03-09,AAPL,259.88,38218533,0.009400,260.0375,260.0375,1
4,2026-03-10,AAPL,260.83,30590765,0.003656,260.1960,260.1960,0
...,...,...,...,...,...,...,...,...
97,2026-07-23,AAPL,321.66,40840778,-0.012980,327.1240,311.4920,1
98,2026-07-24,AAPL,333.02,47489415,0.035317,326.9800,314.3855,1
99,2026-07-27,AAPL,336.91,49604297,0.011681,329.0440,317.0420,1
100,2026-07-28,AAPL,340.08,51859042,0.009409,331.5120,319.9590,0


In [14]:
df['daily_return'].fillna(df['daily_return'].median(), inplace=True)

In [5]:
df['daily_return'].ffill(df['daily_return'].median(),inplace=True)

TypeError: NDFrame.ffill() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given

In [25]:
# prepare features and target
X = df.drop(columns=['target_direction', 'price_date'])
target = df['target_direction']

# numeric and categorical column lists
num_col = X.select_dtypes(include='number').columns.tolist()
cat_col = X.select_dtypes(include=['object', 'category']).columns.tolist()

In [33]:
df['target_direction'].value_counts()

target_direction
1    55
0    44
Name: count, dtype: Int64

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.2, random_state=42)

num_pipeline = Pipeline(steps=[('impute',SimpleImputer(strategy='median')),('scaler', StandardScaler())])
cat_pipeline = Pipeline(steps=[('impute',SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[('num', num_pipeline, num_col),('cat', cat_pipeline, cat_col)])

model  = RandomForestClassifier(n_estimators=100, random_state=42)
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),('model', model)])

model_pipeline.fit(X_train, y_train)
model_predict = model_pipeline.predict(X_test)

print(classification_report(y_test, model_predict))
print(confusion_matrix(y_test, model_predict))

              precision    recall  f1-score   support

         0.0       0.69      0.82      0.75        11
         1.0       0.71      0.56      0.62         9

    accuracy                           0.70        20
   macro avg       0.70      0.69      0.69        20
weighted avg       0.70      0.70      0.69        20

[[9 2]
 [4 5]]


In [34]:
# df.dropna()
# df.drop_duplicates()
# df['daily_return'].ffill(df['daily_return'].median(),inplace=True)

# # prepare features and target
# X = df.drop(columns=['target_direction', 'price_date'])
# target = df['target_direction']

# # numeric and categorical column lists
# num_col = X.select_dtypes(include='number').columns.tolist()
# cat_col = X.select_dtypes(include=['object', 'category']).columns.tolist()

# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report, confusion_matrix

# X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.2, random_state=42)

# num_pipeline = Pipeline(steps=[('impute',SimpleImputer(strategy='median')),('scaler', StandardScaler())])
# cat_pipeline = Pipeline(steps=[('impute',SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(handle_unknown='ignore'))])

# preprocessor = ColumnTransformer(transformers=[('num', num_pipeline, num_col),('cat', cat_pipeline, cat_col)])

# model  = RandomForestClassifier(n_estimators=100, random_state=42)
# model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),('model', model)])

# model_pipeline.fit(X_train, y_train)
# model_predict = model_pipeline.predict(X_test)

# print(classification_report(y_test, model_predict))
# print(confusion_matrix(y_test, model_predict))

# it got me :

#          precision    recall  f1-score   support

#          0.0       0.69      0.82      0.75        11
#          1.0       0.71      0.56      0.62         9

#     accuracy                           0.70        20
#    macro avg       0.70      0.69      0.69        20
# weighted avg       0.70      0.70      0.69        20

# [[9 2]
#  [4 5]]


# from sklearn.model_selection import GridSearchCV

# model_params = {
#     'model__n_estimators': [100, 200, 300],
#     'model__max_depth': [None, 10, 20, 30],
#     'model__min_samples_split': [2, 5, 10],
#     'model__min_samples_leaf': [1, 2, 4],
#     'model__bootstrap': [True, False]
# }

# grid_search = GridSearchCV(model_pipeline, model_params, cv=5)
# grid_search.fit(X_train, y_train)

# print("Best parameters found: ", grid_search.best_params_)
# print("Best cross-validation score: ", grid_search.best_score_)


# Best parameters found:  {'model__bootstrap': True, 'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}
# Best cross-validation score:  0.5183333333333333


In [29]:
from sklearn.model_selection import GridSearchCV

model_params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__bootstrap': [True, False]
}

grid_search = GridSearchCV(model_pipeline, model_params, cv=5)
grid_search.fit(X_train, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

Best parameters found:  {'model__bootstrap': True, 'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Best cross-validation score:  0.5183333333333333


c:\Users\abhay\anaconda3\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


In [35]:
from xgboost import XGBClassifier

model = XGBClassifier(n_estimators=100, random_state=42)
model_xgb_pip = Pipeline(steps=[('preprocessor', preprocessor),('model', model)])
model_xgb_pip.fit(X_train, y_train)
predict_xgb = model_xgb_pip.predict(X_test)

print(classification_report(y_test, predict_xgb))
print(confusion_matrix(y_test, predict_xgb))

print("--------------------------------------------")

from sklearn.model_selection import GridSearchCV
model_params_xgb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__subsample': [0.8, 1.0],
}

grid_search_xgb = GridSearchCV(model_xgb_pip, model_params_xgb, cv=5)
grid_search_xgb.fit(X_train, y_train)
print("Best parameters found: ", grid_search_xgb.best_params_)
print("Best cross-validation score: ", grid_search_xgb.best_score_)


              precision    recall  f1-score   support

         0.0       0.62      0.73      0.67        11
         1.0       0.57      0.44      0.50         9

    accuracy                           0.60        20
   macro avg       0.59      0.59      0.58        20
weighted avg       0.60      0.60      0.59        20

[[8 3]
 [5 4]]
--------------------------------------------
Best parameters found:  {'model__learning_rate': 0.2, 'model__max_depth': 3, 'model__n_estimators': 200, 'model__subsample': 0.8}
Best cross-validation score:  0.5441666666666667
